# Preprocessing the ASSISTments dataset to filter missing and rare values

In [1]:
## Import dataset

import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

path = "/content/drive/MyDrive/education-ml-research/ASSISTments2009/skill_builder_data.csv"

df = pd.read_csv(path,  encoding='latin1')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_9173/1150059154.py:10: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path,  encoding='latin1')


## Remove rows with missing values

In [2]:
# Record original dataset size
original_rows = len(df)

# Count missing values in key columns
print("Missing values before cleaning:")
print(df[["user_id", "problem_id", "skill_id", "correct"]].isnull().sum())

# Number of rows with missing skill IDs
missing_skill_rows = df["skill_id"].isnull().sum()

# Remove rows missing any required information
required_columns = [
    "user_id",
    "problem_id",
    "skill_id",
    "correct"
]

df = df.dropna(subset=required_columns)

# Statistics after cleaning
remaining_rows = len(df)
rows_removed = original_rows - remaining_rows

print("\nCleaning Summary")
print("----------------")
print(f"Original rows: {original_rows:,}")
print(f"Rows removed: {rows_removed:,}")
print(f"Remaining rows: {remaining_rows:,}")
print(f"Percent removed: {100 * rows_removed / original_rows:.2f}%")

Missing values before cleaning:
user_id           0
problem_id        0
skill_id      66326
correct           0
dtype: int64

Cleaning Summary
----------------
Original rows: 525,534
Rows removed: 66,326
Remaining rows: 459,208
Percent removed: 12.62%


## Remove students with fewer than 10 attempts

In [3]:
# Record original number of students
original_students = df["user_id"].nunique()

# Count attempts for each student
attempt_counts = df.groupby("user_id").size()

# Keep only students with at least 10 attempts
valid_students = attempt_counts[attempt_counts >= 10].index

df = df[df["user_id"].isin(valid_students)]

# Statistics after filtering
remaining_students = df["user_id"].nunique()
students_removed = original_students - remaining_students

print("\nStudent Filtering Summary")
print("-------------------------")
print(f"Original students: {original_students:,}")
print(f"Students removed: {students_removed:,}")
print(f"Remaining students: {remaining_students:,}")

## Rows before student filtering but after handling missing values
original_rows = remaining_rows

## Rows after student filtering
remaining_rows = len(df)

print(f"Rows remaining: {remaining_rows:,}")
print(f"Percent of rows retained: {100 * remaining_rows / original_rows:.2f}%")


Student Filtering Summary
-------------------------
Original students: 4,163
Students removed: 1,044
Remaining students: 3,119
Rows remaining: 454,232
Percent of rows retained: 98.92%


## Filter out skills with fewer than 100 attempts

In [4]:
skill_counts = df["skill_id"].value_counts()

print("Number of unique skills:", len(skill_counts))

# How many skills have fewer than 100 interactions?
rare_skills = skill_counts[skill_counts < 100]

print(f"\nSkills with fewer than 100 interactions: {len(rare_skills)}")
print(f"Skills with at least 100 interactions: {len(skill_counts) - len(rare_skills)}")

# Record original statistics
original_rows = len(df)
original_skills = df["skill_id"].nunique()

# Keep only sufficiently common skills
valid_skills = skill_counts[skill_counts >= 100].index

df = df[df["skill_id"].isin(valid_skills)]

# Statistics after filtering
remaining_rows = len(df)
remaining_skills = df["skill_id"].nunique()

print("\nSkill Filtering Summary")
print("-----------------------")
print(f"Original skills: {original_skills}")
print(f"Remaining skills: {remaining_skills}")
print(f"Skills removed: {original_skills - remaining_skills}")

print()

print(f"Original rows: {original_rows:,}")
print(f"Remaining rows: {remaining_rows:,}")
print(f"Rows removed: {original_rows - remaining_rows:,}")
print(f"Percent of rows retained: {100 * remaining_rows / original_rows:.2f}%")

Number of unique skills: 123

Skills with fewer than 100 interactions: 16
Skills with at least 100 interactions: 107

Skill Filtering Summary
-----------------------
Original skills: 123
Remaining skills: 107
Skills removed: 16

Original rows: 454,232
Remaining rows: 453,548
Rows removed: 684
Percent of rows retained: 99.85%


## Final student filter

In [5]:
attempt_counts_final = df.groupby("user_id").size()

valid_students_final = attempt_counts_final[
    attempt_counts_final >= 10
].index

students_before = df["user_id"].nunique()

df = df[df["user_id"].isin(valid_students_final)]

students_after = df["user_id"].nunique()

print("Final Student Filtering")
print("-----------------------")
print(f"Students before: {students_before:,}")
print(f"Students removed: {students_before - students_after:,}")
print(f"Students remaining: {students_after:,}")

print()

print(
    "Students with <10 attempts:",
    (df.groupby("user_id").size() < 10).sum()
)

Final Student Filtering
-----------------------
Students before: 3,116
Students removed: 3
Students remaining: 3,113

Students with <10 attempts: 0


## Reencode the skills and save the dataset

In [6]:
df["skill_encoded"] = (
    df["skill_id"]
      .astype("category")
      .cat.codes
)

print("Encoded skills:", df["skill_encoded"].nunique())

output_path = "/content/drive/MyDrive/education-ml-research/ASSISTments2009/assistments_cleaned.csv"

df.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully!")

Encoded skills: 107
Cleaned dataset saved successfully!


## Importing the cleaned dataset

In [7]:
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

cleaned_path = "/content/drive/MyDrive/education-ml-research/ASSISTments2009/assistments_cleaned.csv"

df_clean = pd.read_csv(cleaned_path, encoding='latin1')

print("Dataset loaded successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset loaded successfully!


## Validating the cleaned dataset

In [18]:
## Check shape
print("Rows:", len(df_clean))
print("Columns:", len(df_clean.columns))
print("")

## Check missing values
required_columns = [
    "user_id",
    "problem_id",
    "skill_encoded",
    "correct",
    "order_id"
]

print(df_clean[required_columns].isnull().sum())
print("")

## Check students that have < 10 attempts
attempts_per_student = df_clean.groupby("user_id").size()
print(
    "Students with <10 attempts:",
    (attempts_per_student < 10).sum()
)
print("")

## Check skills that have < 100 attempts
skill_counts_clean = df_clean["skill_id"].value_counts()

print(
    "Skills with <100 interactions:",
    (skill_counts_clean < 100).sum()
)
print("")

## Check encoding

print(
    "Number of encoded skills:",
    df_clean["skill_encoded"].nunique()
)
print("")
## Check unique students and skills

print("Students:", df_clean["user_id"].nunique())
print("Skills:", df_clean["skill_id"].nunique())
print("Problems:", df_clean["problem_id"].nunique())
print("")

## Check correct values
print(df_clean["correct"].unique())
print("")

print("===== CLEAN DATA SUMMARY =====")
print(f"Interactions: {len(df_clean):,}")
print(f"Students: {df_clean['user_id'].nunique():,}")
print(f"Skills: {df_clean['skill_encoded'].nunique():,}")
print(f"Problems: {df_clean['problem_id'].nunique():,}")
print(f"Missing values: {df_clean[required_columns].isnull().sum().sum()}")

Rows: 453537
Columns: 31

user_id          0
problem_id       0
skill_encoded    0
correct          0
order_id         0
dtype: int64

Students with <10 attempts: 0

Skills with <100 interactions: 0

Number of encoded skills: 107

Students: 3113
Skills: 107
Problems: 17585

[1 0]

===== CLEAN DATA SUMMARY =====
Interactions: 453,537
Students: 3,113
Skills: 107
Problems: 17,585
Missing values: 0


## Create SAKT-ready dataframe

In [20]:
sakt_columns = [
    "user_id",
    "problem_id",
    "skill_encoded",
    "correct",
    "order_id"
]

df_sakt = df_clean[sakt_columns].copy()

print(df_sakt.head())
print(df_sakt.isnull().sum())

sakt_path = "/content/drive/MyDrive/education-ml-research/ASSISTments2009/assistments_sakt_ready.csv"

df_sakt.to_csv(sakt_path, index=False)

print("SAKT-ready dataset saved!")

   user_id  problem_id  skill_encoded  correct  order_id
0    64525       51424              0        1  33022537
1    64525       51435              0        1  33022709
2    70363       51444              0        0  35450204
3    70363       51395              0        1  35450295
4    70363       51481              0        0  35450311
user_id          0
problem_id       0
skill_encoded    0
correct          0
order_id         0
dtype: int64
SAKT-ready dataset saved!


## Data Preprocessing

This study uses the ASSISTments 2009 Skill Builder dataset, a large-scale educational interaction dataset containing student responses to mathematics problems. Prior to model training, the dataset was cleaned and filtered to ensure that all interactions used for knowledge tracing contained reliable information about the student, the tested skill, and the learning outcome.

First, interactions without skill annotations were removed. Approximately 12% of records lacked both a `skill_id` and corresponding `skill_name`, meaning that these interactions could not be associated with a specific mathematical concept. Since knowledge tracing models learn concept-level representations and update student knowledge states based on skill interactions, these records were excluded.

Next, student interaction histories were filtered to remove extremely sparse sequences. Students with fewer than 10 recorded interactions were removed because knowledge tracing models rely on sequential learning histories to estimate evolving knowledge states. Short sequences provide insufficient information about a student's learning trajectory and can introduce instability during training and evaluation. Although this filtering removed approximately 25% of students, these students represented only approximately 2% of all interaction records, preserving the majority of available learning data.

The dataset was then filtered at the skill level. Skills with fewer than 100 recorded interactions were removed because extremely rare skills provide limited training examples and may result in unreliable skill representations. This step reduced the number of skills from 123 to 107 while maintaining sufficient data coverage for the remaining concepts.

After all filtering operations, student sequence lengths were re-evaluated to ensure that every student retained at least 10 interactions. This additional filtering step was necessary because removing rare skills could reduce the number of remaining interactions for individual students.

Finally, skill identifiers were converted into consecutive integer encodings to create numerical representations suitable for neural network models. The final cleaned dataset contained 453,537 interactions from 3,113 students across 107 mathematical skills and 17,585 problems. The resulting dataset will be used as the input for the knowledge tracing models evaluated in this study.